# Patient Medications Table - Population & Normalization

**Project:** Medical Data Cleaning  
**Author:** Tesneem Fnais  
**Date:** April 2026

## Purpose

The original `medical_data` table stored medications as **comma-separated strings** in a single column (e.g., `"Metformin, Lisinopril"` paired with dosages `"500, 10"`). This violates first normal form and makes the data hard to query.

This notebook normalizes that data into a separate `patient_medications` table -one row per medication per patient- with dosage values and units cleanly separated.

## Setup

In [28]:
# pip install mysql-connector-python

In [42]:
# import libraries for this task

import mysql.connector
import pandas as pd

In [44]:
# connect to MySQL using getpass to avoid exposing the password
import getpass

con = mysql.connector.connect(
    host = 'localhost',
    user = 'root',
    password = getpass.getpass('MySQL password: '),
    database = 'medical_cleaning')

cursor = con.cursor()
print('Connected Successfully :)')

MySQL password:  ········


Connected Successfully :)


## Step 1: Load the medications data

Pull `patient_id`, `medications`, and `dosage_mg` from the cleaned `medical_data` table. 
Each patient may have multiple medications stored as comma-separated values, these need to be split out into individual rows.

In [46]:
# read medical_data into a dataframe

query = 'SELECT patient_id, medications, dosage_mg FROM medical_data'
df = pd.read_sql(query, con)

df

C:\Users\tasne\AppData\Local\Temp\ipykernel_13744\2372397210.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, con)


,patient_id,medications,dosage_mg
0,P001,"metformin, lisinopril","500, 10"
1,P002,amlodipine,5
2,P003,"amoxicillin, paracetamol","1000, 500"
3,P004,"azithromycin, ceftriaxone","500, 1000"
4,P005,ors,N/A
5,P006,"sumatriptan, ibuprofen","50, 400"
6,P007,tramadol,50
7,P008,"insulin, metformin","40units, 1000"
8,P009,"tamsulosin, ketorolac","0.4, 30"
9,P010,"iron supplements, folic acid","200, 5"


## Step 2: Split medications into individual rows

Convert each comma-separated string into individual entries, pairing each medication with its corresponding dosage using `zip()`. 
The result has one row per (patient, medication) pair.

In [49]:
# parse each row

rows = []

for _, row in df.iterrows():
    
    meds = [m.strip() for m in str(row['medications']).split(',')]
    
    dosages = [d.strip() for d in str(row['dosage_mg']).split(',')]
    
    for med, dos in zip(meds, dosages):
        rows.append({
        'patient_id': row['patient_id'],
        'medication': med,
        'dosage_raw': dos
    })

parsed_df = pd.DataFrame(rows)
parsed_df

,patient_id,medication,dosage_raw
0,P001,metformin,500
1,P001,lisinopril,10
2,P002,amlodipine,5
3,P003,amoxicillin,1000
4,P003,paracetamol,500
5,P004,azithromycin,500
6,P004,ceftriaxone,1000
7,P005,ors,N/A
8,P006,sumatriptan,50
9,P006,ibuprofen,400


## Step 3: Separate dosage value from unit

Dosage entries come in mixed formats: `"500"` (assumed mg), `"40units"`, `"100mcg"`, and `"N/A"` (no dosage applicable). 
A small regex extracts the numeric part and the unit separately, defaulting to `mg` when no unit is specified.

In [52]:
import re

# Function to extract numeric dosage and unit separately
def parse_dosage(dosage_raw):
    # re.match looks for a number (including decimals) at the start of the string
    match = re.match(r'([\d.]+)([a-zA-Z]*)', str(dosage_raw).strip())
    
    if dosage_raw == 'N/A':
        return None, 'Dosage Not Applicable'  # no dosage, no unit
    elif match:
        value = match.group(1)   # the number part e.g. '40'
        unit = match.group(2)    # the unit part e.g. 'units', 'mg', 'mcg'
        unit = unit if unit else 'mg'  # default to mg if no unit specified
        return float(value), unit
    else:
        return None, None

# Apply the function to each row in dosage_raw column
parsed_df[['dosage', 'dosage_unit']] = parsed_df['dosage_raw'].apply(
    lambda x: pd.Series(parse_dosage(x))
)

parsed_df

,patient_id,medication,dosage_raw,dosage,dosage_unit
0,P001,metformin,500,500.0,mg
1,P001,lisinopril,10,10.0,mg
2,P002,amlodipine,5,5.0,mg
3,P003,amoxicillin,1000,1000.0,mg
4,P003,paracetamol,500,500.0,mg
5,P004,azithromycin,500,500.0,mg
6,P004,ceftriaxone,1000,1000.0,mg
7,P005,ors,N/A,NaN,Dosage Not Applicable
8,P006,sumatriptan,50,50.0,mg
9,P006,ibuprofen,400,400.0,mg


## Step 4: Prepare for MySQL insertion

MySQL expects `NULL` rather than pandas' `NaN`. Replace any `NaN` values with Python `None`, which the connector translates to `NULL`. Also drop the now-redundant `dosage_raw` column.

In [55]:
# Replace NaN in the 'dosage' column specifically (numeric)
parsed_df['dosage'] = parsed_df['dosage'].where(parsed_df['dosage'].notna(), None)

# Drop the dosage_raw column since we don't need it anymore
parsed_df = parsed_df.drop(columns=['dosage_raw'])

# Preview the result
parsed_df

,patient_id,medication,dosage,dosage_unit
0,P001,metformin,500.0,mg
1,P001,lisinopril,10.0,mg
2,P002,amlodipine,5.0,mg
3,P003,amoxicillin,1000.0,mg
4,P003,paracetamol,500.0,mg
5,P004,azithromycin,500.0,mg
6,P004,ceftriaxone,1000.0,mg
7,P005,ors,NaN,Dosage Not Applicable
8,P006,sumatriptan,50.0,mg
9,P006,ibuprofen,400.0,mg


In [57]:
# Extra safety net: replace any remaining NaN across all columns
import numpy as np

# Replace NaN with None
parsed_df = parsed_df.replace({np.nan: None})

# Preview
parsed_df

,patient_id,medication,dosage,dosage_unit
0,P001,metformin,500.0,mg
1,P001,lisinopril,10.0,mg
2,P002,amlodipine,5.0,mg
3,P003,amoxicillin,1000.0,mg
4,P003,paracetamol,500.0,mg
5,P004,azithromycin,500.0,mg
6,P004,ceftriaxone,1000.0,mg
7,P005,ors,None,Dosage Not Applicable
8,P006,sumatriptan,50.0,mg
9,P006,ibuprofen,400.0,mg


## Step 5: Insert into the `patient_medications` table

Loop through each row and insert it into the empty `patient_medications` table created in MySQL. Using parameterized queries (`%s` placeholders) protects against SQL injection. `con.commit()` finalizes the transaction.

In [60]:
# Loop through each row in parsed_df and insert into MySQL
for _, row in parsed_df.iterrows():
    cursor.execute("""
        INSERT INTO patient_medications (patient_id, medication, dosage, dosage_unit)
        VALUES (%s, %s, %s, %s)
    """, (row['patient_id'], row['medication'], row['dosage'], row['dosage_unit']))

# Commit the changes to the database
con.commit()

print(f"{len(parsed_df)} rows inserted successfully!")

48 rows inserted successfully!


In [62]:
# Verify the insert
cursor.execute("SELECT COUNT(*) FROM patient_medications")
print(f"Total rows in patient_medications: {cursor.fetchone()[0]}")

cursor.execute("SELECT * FROM patient_medications LIMIT 5")
for row in cursor.fetchall():
    print(row)

Total rows in patient_medications: 48
(1, 'P001', 'metformin', Decimal('500.00'), 'mg')
(2, 'P001', 'lisinopril', Decimal('10.00'), 'mg')
(3, 'P002', 'amlodipine', Decimal('5.00'), 'mg')
(4, 'P003', 'amoxicillin', Decimal('1000.00'), 'mg')
(5, 'P003', 'paracetamol', Decimal('500.00'), 'mg')


## Done :)

The `patient_medications` table is now populated with one row per medication. From here, queries like *"average dosage of Metformin across patients"* or *"patients on more than 3 medications"* become straightforward, which wasn't possible with the original comma-separated format.